# PINNs for Mechanical Engineers — Tutorial 8: The Inverse Problem
### Where PINNs finally win: recovering unknown physics from a handful of sensors

*📺 Video on YouTube | 🐍 Python · PyTorch | Intermediate*

*Notebook repository: [github.com/rraj-io/PINN_for_Mechanical_Engineer](https://github.com/rraj-io/PINN_for_Mechanical_Engineer)*

---

## Series Roadmap

Seven tutorials, seven **forward** problems: given the physics and the boundary conditions, find the field. And seven honest verdicts — a classical solver was usually faster and more accurate.

This one flips the arrow. We don't know the physics. We have a few noisy thermometer readings and a governing equation with a **missing coefficient**, and we want both the full field *and* the coefficient. This is the problem PINNs are genuinely, categorically better at.

| # | Tutorial | Focus | Status |
|---|----------|-------|--------|
| 1–5 | Cavity · Heat · Elasticity · Convection · Beam | forward solving, one physics at a time | ✅ |
| 6–7 | Backward-Facing Step (steady, then transient) | separation; time-dependent flow | ✅ |
| **8** | **Inverse Problem — parameter identification** | **learning unknown physics from data** | **📍 You are here** |

> **New to PINNs?** This series assumes the fundamentals: autograd, collocation points, the residual loss, Adam → L-BFGS. Starting from zero? The 100+ page ebook builds all of it from the first autograd call.
>
> 👉 **[Get *PINNs from Scratch* on Gumroad](https://rajrohit403.gumroad.com/l/pinn_tutorial)**

---

## 1 — What You Will Learn

By the end of this notebook you will have:

- Solved your first **inverse problem**: recovering an unknown material property — the thermal conductivity $k$ — from **12 noisy sensor readings** and the governing equation alone
- Seen the entire trick, which is one line: make the unknown a **`nn.Parameter`** and hand it to the optimizer alongside the network weights
- Added a third loss term — the **data misfit** — and understood how physics and measurements cooperate: the sensors say *what*, the PDE says *how*
- Recovered $k$ to **within ~1%** of truth while simultaneously reconstructing the **full temperature field** at every point, including everywhere you never measured
- Learned why **noise sets the accuracy floor**, and watched the recovered parameter carry a small honest bias because of it
- Understood why this is the tutorial where PINNs beat classical solvers: an FEM code needs a whole **adjoint framework** to do this; a PINN needs `requires_grad=True`

**Prerequisites:** Tutorial 2 (steady heat conduction). The forward machinery is identical — a scalar field, second derivatives, Dirichlet walls. Everything new is in the *unknown*.

⏱️ **Runtime:** quick. A 2D scalar problem with a small network — a few minutes on CPU, well under a minute on GPU.

## 2 — Forward vs. inverse, and why this one matters

Every previous tutorial asked the **forward** question:

> *Given the physics and the boundary conditions, what is the field?*

Real engineering constantly asks the **inverse** question instead:

> *I have some measurements. What is the physics?*

You have thermocouples in a wall but don't know the insulation's conductivity. You measure a few pressures but don't know the friction factor. You have thermal images of a component but don't know where the heat is being generated. In every case the unknown isn't the field — it's a **property**, and the sparse data is all you have.

### The problem we'll solve

Steady heat conduction with a known internal heat source $q$, on the unit square, with the plate edges held at $T=0$:

$$k\,\nabla^2 T + q(x,y) = 0,\qquad T=0 \text{ on the boundary}.$$

Here $q$ is known (we're heating the plate ourselves, so we know how hard), and $T$ is unknown as usual — but so is the **thermal conductivity $k$**. All we have are a handful of noisy temperature readings scattered across the plate.

**Two unknowns, one network, one extra parameter.**

### Why a PINN is the right tool

In a classical solver, the field is a mesh of numbers and the parameter is a coefficient in a matrix — inverting for the parameter requires building an **adjoint solver**, deriving sensitivity equations, and running an optimization loop around the whole simulation. It's a research project.

In a PINN, the field is a network and the parameter is a scalar. Both are just tensors with gradients. You add $k$ to the optimizer and it learns alongside the weights. **That's the whole method** — and it's why this is the tutorial where the series' honest verdict finally flips.

### How the sensors and the physics cooperate

The data term alone is hopeless: 12 points cannot determine a field, let alone a coefficient — infinitely many $(T,k)$ pairs fit 12 numbers. The physics term alone is worse: without data, any $k$ works if $T$ rescales to match.

Together they're rigid. The PDE constrains the *shape* of the field everywhere; the sensors fix its *scale* at a few places. Only one $(T,k)$ satisfies both — that's the mechanism, and it's why sparse data suffices.

## 3 — Setup

Three libraries, a seed, device selection, and the "truth" we're going to pretend not to know.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import math, time

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
PI = math.pi

# --- The ground truth (the notebook "knows" this only to generate data and to score us) ---
K_TRUE = 4.7          # the thermal conductivity we are pretending not to know
N_SENSORS = 12        # how many noisy readings we get
NOISE = 0.02          # 2% measurement noise

print(f"hidden truth: k = {K_TRUE}   |   {N_SENSORS} sensors, {NOISE*100:.0f}% noise")

## 4 — The synthetic experiment

To *test* an inverse method you need a case where you know the answer. So we manufacture one: choose a temperature field, then choose the heat source that would produce it under the true conductivity.

Take the field to be

$$T(x,y)=\sin(\pi x)\sin(\pi y),$$

which conveniently vanishes on all four edges. Substituting into $k\nabla^2T+q=0$ gives the source that must have created it:

$$q(x,y)=2k_{\text{true}}\pi^2\sin(\pi x)\sin(\pi y).$$

This is the **method of manufactured solutions**, and it's the honest way to validate an inverse solver: we know both the field and the parameter, so we can score the recovery exactly.

Note the structure — $q$ carries $k_{\text{true}}$ inside it. That's precisely what makes $k$ recoverable: the *ratio* of source to curvature encodes the conductivity, and the sensors pin down which ratio is real.

In [ ]:
def T_exact(x, y):
    "The manufactured true temperature field."
    return torch.sin(PI*x)*torch.sin(PI*y)

def source(x, y):
    "The heat source that produces T_exact under k_true. Known to us (we apply it)."
    return 2.0*K_TRUE*PI**2*torch.sin(PI*x)*torch.sin(PI*y)

# sanity: does the truth satisfy the PDE?
xc = torch.rand(5,1,device=device,requires_grad=True)
yc = torch.rand(5,1,device=device,requires_grad=True)
T  = T_exact(xc,yc)
Tx = torch.autograd.grad(T,xc,torch.ones_like(T),create_graph=True)[0]
Ty = torch.autograd.grad(T,yc,torch.ones_like(T),create_graph=True)[0]
lap = (torch.autograd.grad(Tx,xc,torch.ones_like(Tx),create_graph=True)[0]
     + torch.autograd.grad(Ty,yc,torch.ones_like(Ty),create_graph=True)[0])
resid = K_TRUE*lap + source(xc,yc)
print(f"residual of the manufactured truth: {resid.abs().max().item():.2e}   (want ~0)")
print(f"T on edges: {T_exact(torch.zeros(1,1),torch.rand(1,1)).item():.2e}  (want 0)")

## 5 — The measurements

Twelve thermometers dropped at random positions, each reading the true temperature plus 2% noise. **This is all the data we get** — everything else must come from the physics.

In [ ]:
gen = torch.Generator().manual_seed(42)

x_s = torch.rand(N_SENSORS,1,generator=gen).to(device)
y_s = torch.rand(N_SENSORS,1,generator=gen).to(device)
T_obs = T_exact(x_s,y_s) + NOISE*torch.randn(N_SENSORS,1,generator=gen).to(device)

print(f"{N_SENSORS} sensors placed. First few readings:")
for i in range(4):
    print(f"  ({x_s[i].item():.3f}, {y_s[i].item():.3f})  T_measured = {T_obs[i].item():+.4f}  "
          f"(true {T_exact(x_s[i],y_s[i]).item():+.4f})")

## 6 — Collocation points

Identical to Tutorial 2: interior points where we enforce the PDE, and boundary points where we enforce $T=0$. The sensors are *extra* — they don't replace the physics, they anchor it.

In [ ]:
def rb(n): return torch.rand(n,1,generator=gen).to(device)

N_f, N_b = 1500, 100
x_f = rb(N_f).requires_grad_(True)
y_f = rb(N_f).requires_grad_(True)

x_b, y_b = rb(N_b), rb(N_b)
zeros = torch.zeros(N_b,1,device=device); ones = torch.ones(N_b,1,device=device)

print(f"interior collocation: {N_f} | boundary: {4*N_b} | sensors: {N_SENSORS}")

In [ ]:
# visualise: physics everywhere, data at a few points
plt.figure(figsize=(5.6,5.6))
plt.scatter(x_f.detach().cpu(), y_f.detach().cpu(), s=3, alpha=0.25, color="#2A9D8F",
            label=f"collocation — PDE enforced ({N_f})")
plt.scatter(torch.cat([zeros,ones,x_b,x_b]).cpu(), torch.cat([y_b,y_b,zeros,ones]).cpu(),
            s=9, color="#1A1A2E", label=f"boundary: T=0 ({4*N_b})")
plt.scatter(x_s.cpu(), y_s.cpu(), s=150, color="#E76F51", marker="*", zorder=5,
            edgecolors='k', linewidths=0.5, label=f"sensors — the only data ({N_SENSORS})")
plt.xlabel("x"); plt.ylabel("y"); plt.gca().set_aspect("equal")
plt.title("Physics everywhere, measurements almost nowhere")
plt.legend(loc="upper center", bbox_to_anchor=(0.5,-0.10), fontsize=8)
plt.tight_layout(); plt.show()

## 7 — The network *and the unknown parameter*

Here is the entire inverse method.

The network is ordinary — a small MLP mapping $(x,y)\to T$, exactly as in Tutorial 2. What's new is the line after it: the unknown conductivity is declared as a **`nn.Parameter`**, which means PyTorch will compute a gradient for it and the optimizer will update it just like a weight.

One detail worth the trouble: we learn $\log k$ rather than $k$. Conductivity must be positive, and $k=e^{\log k}$ enforces that automatically — no clamping, no risk of the optimizer wandering into negative (unphysical) territory.

We initialise at $\log k = 0$, i.e. **$k=1.0$** — a deliberately wrong first guess, well away from the true 4.7.

In [ ]:
class PINN(nn.Module):
    "Ordinary network: (x,y) -> T."
    def __init__(self, n_hidden=48, n_layers=4):
        super().__init__()
        layers = [nn.Linear(2,n_hidden), nn.Tanh()]
        for _ in range(n_layers-1):
            layers += [nn.Linear(n_hidden,n_hidden), nn.Tanh()]
        layers += [nn.Linear(n_hidden,1)]
        self.net = nn.Sequential(*layers)
        for m in self.net:
            if isinstance(m,nn.Linear):
                nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)
    def forward(self,x,y):
        return self.net(torch.cat([x,y],dim=1))

model = PINN().to(device)

# ---- THE inverse ingredient: the unknown is a trainable parameter ----
log_k = nn.Parameter(torch.tensor(0.0, device=device))    # k = exp(0) = 1.0 to start
def k_value(): return torch.exp(log_k)

print(f"network parameters: {sum(p.numel() for p in model.parameters())}")
print(f"unknown physical parameter: k = {k_value().item():.4f}  (initial guess; truth is {K_TRUE})")

## 8 — The residual, with the unknown inside it

The PDE residual is Tutorial 2's, with one difference: the conductivity is no longer a constant you typed in — it's `k_value()`, a **live tensor with a gradient**. Every backward pass therefore tells us not only how to improve the field, but how to improve $k$.

In [ ]:
def grad(o,i):
    return torch.autograd.grad(o,i,torch.ones_like(o),create_graph=True)[0]

def residual(x,y):
    "k * lap(T) + q  -- with k a TRAINABLE parameter."
    T  = model(x,y)
    Tx, Ty = grad(T,x), grad(T,y)
    lap = grad(Tx,x) + grad(Ty,y)
    return k_value()*lap + source(x,y)

r = residual(x_f[:64], y_f[:64])
print(f"residual shape {tuple(r.shape)} | mean|r| untrained: {r.abs().mean().item():.3e}")
print(f"does k receive a gradient? {log_k.requires_grad}")

## 9 — The loss: physics + boundaries + data

Three terms. The first two are Tutorial 2's. The third is the new one, and it's the bridge between measurement and model:

$$\mathcal L=\underbrace{\lVert k\nabla^2T+q\rVert^2}_{\text{physics}}
+\lambda_b\underbrace{\lVert T\rVert^2_{\partial\Omega}}_{\text{boundaries}}
+\lambda_d\underbrace{\lVert T(x_s,y_s)-T_{\text{obs}}\rVert^2}_{\textbf{data misfit}}$$

The data weight is large ($\lambda_d=100$) because there are only twelve of those points and they carry all the information about scale. The residual is divided by $k_{\text{true}}^2$ purely to keep it numerically comparable to the other terms — a scaling convenience, not physics.

In [ ]:
W_BC, W_DATA = 10.0, 100.0
def mse(z): return (z**2).mean()

def compute_losses():
    loss_pde = mse(residual(x_f,y_f)) / (K_TRUE**2)        # scaled for comparability
    loss_bc  = (mse(model(zeros,y_b)) + mse(model(ones,y_b))
              + mse(model(x_b,zeros)) + mse(model(x_b,ones)))
    loss_data = mse(model(x_s,y_s) - T_obs)                 # <- the sensors
    return loss_pde, loss_bc, loss_data

def total_loss():
    lp,lb,ld = compute_losses()
    return lp + W_BC*lb + W_DATA*ld

lp,lb,ld = compute_losses()
print(f"untrained: pde={lp.item():.3e}  bc={lb.item():.3e}  data={ld.item():.3e}")

## 10 — Scoring the recovery

Two numbers matter: how close the recovered $k$ is to the truth, and how well the reconstructed field matches everywhere — including the vast majority of the plate where we never measured anything.

In [ ]:
ng = 60
gx = torch.linspace(0,1,ng,device=device)
GX,GY = torch.meshgrid(gx,gx,indexing='ij')
gxr,gyr = GX.reshape(-1,1), GY.reshape(-1,1)
T_ref = T_exact(gxr,gyr)

def field_error():
    with torch.no_grad(): Tp = model(gxr,gyr)
    return (torch.linalg.norm(Tp-T_ref)/torch.linalg.norm(T_ref)).item()

def k_error():
    return abs(k_value().item()-K_TRUE)/K_TRUE*100

print(f"untrained:  k = {k_value().item():.4f} ({k_error():.1f}% off)   field error = {field_error():.3e}")

## 11 — Training: the field and the parameter, together

One optimizer, two kinds of unknown. Note the parameter list — `model.parameters()` **plus** `log_k`. That single concatenation is what makes this an inverse solver.

**Watch the `k` column.** It starts at 1.0 and climbs toward 4.7 as the field takes shape. The field and the parameter improve *together*: a better field sharpens the estimate of $k$, and a better $k$ makes the physics term more accurate.

**Reference run** (real values, CPU, seeds as set here):
```
[    0] pde=9.55e+01 bc=5.77e-01 data=1.11e-01 | k=1.0000  err=6.56e-01
[  500] pde=5.23e+00 bc=1.63e-01 data=5.11e-02 | k=1.8726  err=7.57e-01
[ 1000] pde=8.01e-01 bc=7.21e-02 data=1.75e-02 | k=2.6554  err=4.37e-01
[ 1500] pde=1.68e-01 bc=2.56e-02 data=6.25e-03 | k=3.3088  err=2.53e-01
[ 2000] pde=4.47e-02 bc=9.82e-03 data=2.00e-03 | k=3.8243  err=1.37e-01
[ 2500] pde=1.18e-02 bc=1.53e-03 data=1.13e-03 | k=4.2084  err=8.75e-02
[ 3000] pde=4.99e-03 bc=1.03e-03 data=1.82e-04 | k=4.4715  err=3.01e-02
[ 3500] pde=2.44e-03 bc=3.53e-04 data=7.51e-05 | k=4.6311  err=1.11e-02
[ 4000] pde=1.78e-03 bc=2.37e-04 data=4.59e-05 | k=4.7148  err=9.92e-03
[ 4500] pde=1.41e-03 bc=2.02e-04 data=4.37e-05 | k=4.7488  err=1.26e-02
```

In [ ]:
history = {"it":[], "pde":[], "bc":[], "data":[], "k":[], "err":[]}
def record(it):
    lp,lb,ld = compute_losses()
    history["it"].append(it); history["pde"].append(lp.item())
    history["bc"].append(lb.item()); history["data"].append(ld.item())
    history["k"].append(k_value().item()); history["err"].append(field_error())
    print(f"[{it:5d}] pde={lp.item():.2e} bc={lb.item():.2e} data={ld.item():.2e} "
          f"| k={k_value().item():.4f} (true {K_TRUE})  err={field_error():.2e}")

# the field AND the parameter go into the same optimizer
optimizer = torch.optim.Adam(list(model.parameters()) + [log_k], lr=1e-3)

ITERS = 5000
record(0)
t0 = time.time()
for it in range(1, ITERS+1):
    optimizer.zero_grad(); total_loss().backward(); optimizer.step()
    if it % 250 == 0: record(it)
print(f"\nfinished in {time.time()-t0:.1f}s")
print(f"RECOVERED k = {k_value().item():.4f}   TRUE k = {K_TRUE}   error {k_error():.2f}%")

## 12 — Watching the parameter converge

The left panel is the story: a scalar starting at a wrong guess and walking to the truth, driven by nothing but twelve noisy numbers and a differential equation.

In [ ]:
it = np.array(history["it"])
INK="#1A1A2E"; CORAL="#E76F51"; TEAL="#2A9D8F"; PUR="#6D4C9E"
fig,ax = plt.subplots(1,2,figsize=(13,4.2))

ax[0].plot(it, history["k"], 'o-', color=CORAL, lw=2, ms=3, label='recovered $k$')
ax[0].axhline(K_TRUE, ls='--', color=INK, lw=1.4, label=f'true $k$ = {K_TRUE}')
ax[0].set_xlabel('iteration'); ax[0].set_ylabel('$k$')
ax[0].set_title('The unknown parameter finding itself'); ax[0].legend(fontsize=9)

ax[1].semilogy(it, history["pde"], '-', color=INK, lw=1.5, label='PDE residual')
ax[1].semilogy(it, history["bc"], '--', color=TEAL, lw=1.2, label='boundary')
ax[1].semilogy(it, history["data"], ':', color=CORAL, lw=1.4, label='data misfit')
ax[1].semilogy(it, history["err"], '-.', color=PUR, lw=1.4, label='field error')
ax[1].set_xlabel('iteration'); ax[1].set_ylabel('loss / error')
ax[1].set_title('Physics and data converging together'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 13 — The reconstructed field

The parameter is only half the prize. The network also gives us $T$ **everywhere** — a full field reconstructed from twelve points, because the physics filled in everything between them.

In [ ]:
with torch.no_grad(): Tp = model(gxr,gyr)
P = Tp.reshape(ng,ng).cpu().numpy()
E = T_ref.reshape(ng,ng).cpu().numpy()
D = np.abs(P-E)
Xn,Yn = GX.cpu().numpy(), GY.cpu().numpy()

fig,ax = plt.subplots(1,3,figsize=(15,4.2))
for a,(F,ttl,cm) in zip(ax,[(P,'PINN reconstruction','inferno'),
                            (E,'True field','inferno'),
                            (D,'absolute error','viridis')]):
    cf=a.contourf(Xn,Yn,F,levels=40,cmap=cm); fig.colorbar(cf,ax=a,fraction=0.046)
    a.set_title(ttl); a.set_xlabel('x'); a.set_ylabel('y'); a.set_aspect('equal')
ax[0].scatter(x_s.cpu(),y_s.cpu(),s=90,marker='*',color='cyan',edgecolors='k',linewidths=0.5)
plt.suptitle(f'Full field from {N_SENSORS} sensors (stars) — recovered k = {k_value().item():.3f} vs true {K_TRUE}', y=1.02)
plt.tight_layout(); plt.show()

print(f"field relative L2 error : {field_error():.3e}")
print(f"parameter error         : {k_error():.2f}%")

## 14 — What You Have Built

For seven tutorials the honest verdict was the same: on a clean forward problem, a classical solver wins. Here it flips.

You recovered a material property you were never told, from twelve noisy readings, **to within about 1%** — and got the entire temperature field as a by-product. The method was one line:

```python
log_k = nn.Parameter(torch.tensor(0.0))
optimizer = torch.optim.Adam(list(model.parameters()) + [log_k], lr=1e-3)
```

An FEM code cannot do that. To invert for a coefficient it needs an adjoint solver, hand-derived sensitivity equations, and an optimization loop wrapped around the entire simulation. In a PINN the parameter is simply another tensor with a gradient — inverse problems are not an extension of the method, they're the *same* method.

| | Forward (Tutorials 1–7) | Inverse (this one) |
|---|---|---|
| known | physics, BCs | sparse data, equation form |
| unknown | the field | the field **and** a parameter |
| classical cost | one solve | adjoint framework + optimization loop |
| PINN cost | one training run | **the same training run** |

**The honest footnote.** Notice the recovered $k$ settles a touch above 4.7 rather than landing exactly on it — that residual bias comes from the **2% sensor noise**. Inverse problems inherit the quality of their data; no amount of extra training removes it, because the noise, not the optimizer, sets the floor. Report inverse results with that in mind.

---

### Failure modes to watch for

| Symptom | Cause | Fix |
|---|---|---|
| $k$ drifts to something absurd | data weight too low — physics alone can't fix scale | raise `W_DATA`; add sensors |
| $k$ goes negative (unphysical) | learning $k$ directly instead of $\log k$ | use the exponential parameterisation |
| $k$ converges, field doesn't | too few collocation points | raise `N_f` |
| Field fits sensors, wrong elsewhere | overfitting the data, ignoring physics | lower `W_DATA` — the PDE is your regulariser |
| Recovered $k$ biased | measurement noise — **expected** | more sensors reduces it; noise sets the floor |
| $k$ barely moves | it isn't in the optimizer | check `list(model.parameters()) + [log_k]` |

### Exercises

1. **Sensor budget.** Sweep `N_SENSORS` from 3 to 50 and plot the recovered $k$ error. How few can you get away with? Where do the returns flatten?
2. **Noise study.** Sweep `NOISE` from 0 to 10%. At zero noise, how exact does $k$ become? This isolates the noise floor from the method.
3. **Two unknowns.** Make the source amplitude unknown *as well* as $k$. Are both still recoverable, or is the problem now ill-posed? (Think about what the sensors can distinguish.)
4. **A field, not a constant.** Let $k(x)$ vary in space and represent it with a **second small network**. Now you're recovering an unknown *function* — the real-world version of this problem.
5. **Inverse convection.** Return to Tutorial 6's backward-facing step, hide the Reynolds number, and recover it from a few velocity probes.
6. **Sensor placement.** Instead of random positions, try clustering sensors near the hot centre versus spreading them out. Does placement matter more than count?
